# Cyber expert training

### 1: Setup Environment with uv
Instead of standard pip, we use uv to ensure we follow the uv.lock file you generated on your Mac.

In [13]:
# List all folders in /content to find your project name
!pwd
!ls -al
!cat .config/active_config

/content
total 20
drwxr-xr-x 1 root root 4096 Jan 10 03:36 .
drwxr-xr-x 1 root root 4096 Jan 10 01:38 ..
drwxr-xr-x 4 root root 4096 Dec  9 14:41 .config
-rw-r--r-- 1 root root  519 Jan 10 03:36 pyproject.toml
drwxr-xr-x 1 root root 4096 Dec  9 14:42 sample_data
default

In [ ]:
!git clone https://YOUR_TOKEN@github.com/Bilel-Eljaamii/expert-factory.git
%cd expert-factory

In [14]:
# 1. Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] += ":/root/.cargo/bin"

# 2. Install dependencies into the Colab system environment
# We use --system because Colab handles its own Python path
print("Installing expert-factory dependencies...")
!uv pip install --system .

downloading uv 0.9.24 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Installing expert-factory dependencies...
Using Python 3.12.12 environment at: /usr
Resolved 131 packages in 6.19s                                       
Prepared 51 packages in 47.87s                                           
Uninstalled 23 packages in 1.11s
Installed 51 packages in 548ms                              
 + anthropic==0.46.0
 + bitsandbytes==0.49.1
 + bonito==0.2
 + cfgv==3.5.0
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + distlib==0.4.0
 + docopt==0.6.2
 + experts-factory==0.0.1 (from file:///content)
 + filetype==1.2.0
 + ftfy==6.3.1
 + identify==2.6.15
 + loguru==0.7.3
 + markdown2==2.5.4
 + markdownify==1.2.2
 + marker-pdf==1.10.1
 + msgspec==0.20.0
 + nodeenv==1.10.0
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-c

### 2: import librairies and load config

In [15]:
import yaml
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
/tmp/ipython-input-3684789636.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Load the config
with open("configs/cybersecurity_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# 2. Extract sections
model_conf = config['model']
lora_conf  = config['lora']
train_conf = config['training']

### 2: Initialize Model (Gemma 2 9B)

We load the model in 4-bit to fit into the T4's 16GB VRAM.

In [ ]:
# Load Model using YAML values
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_conf['model_name'],
    max_seq_length = model_conf['max_seq_length'],
    dtype          = model_conf['dtype'],
    load_in_4bit   = model_conf['load_in_4bit'],
)

# Add LoRA Adapters using YAML values
model = FastLanguageModel.get_peft_model(
    model,
    r                          = lora_conf['r'],
    target_modules             = lora_conf['target_modules'],
    lora_alpha                 = lora_conf['lora_alpha'],
    lora_dropout               = lora_conf['lora_dropout'],
    bias                       = lora_conf['bias'],
    use_gradient_checkpointing = lora_conf['use_gradient_checkpointing'],
    random_state               = lora_conf['random_state'],
)

## 4: Load and Format Expert Data

This cell loads the Cybersecurity datasets and applies the Gemma chat template.

In [ ]:
# Loading the Cybersecurity dataset
dataset = load_dataset("ethanolivertroy/nist-cybersecurity-training", split = "train")


# Define the Expert Persona
SYSTEM_PROMPT = """
You are a Senior Cybersecurity Analyst and Threat Hunter. Provide technical, accurate,
and actionable security advice.
"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Apply the Gemma 2 Chat Template
        text = f"""
            <start_of_turn>user\n{SYSTEM_PROMPT}\n\n{instruction}<end_of_turn>\n
            <start_of_turn>model\n{output}<end_of_turn>
            """
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 5: Training Configuration (T4 Optimized)

These settings prevent "Out of Memory" (OOM) errors on the T4 GPU.

In [ ]:
# Extract training sub-dictionary
train_conf = config['training']

# Handle the Hardware Logic (The "not torch.cuda..." part)
# This checks if the GPU supports bf16; if not, it falls back to fp16.
has_bf16 = torch.cuda.is_bf16_supported()

# Define Training Arguments
training_args = TrainingArguments(
    per_device_train_batch_size = train_conf['per_device_train_batch_size'],
    gradient_accumulation_steps = train_conf['gradient_accumulation_steps'],
    warmup_steps                = train_conf['warmup_steps'],
    max_steps                   = train_conf['max_steps'],
    learning_rate               = float(train_conf['learning_rate']),
    fp16                        = not has_bf16, # True on T4
    bf16                        = has_bf16,     # False on T4
    logging_steps               = train_conf['logging_steps'],
    optim                       = train_conf['optim'],
    weight_decay                = train_conf['weight_decay'],
    lr_scheduler_type           = train_conf['lr_scheduler_type'],
    seed                        = train_conf['seed'],
    output_dir                  = train_conf['output_dir'],
    report_to                   = "none", # Prevents unwanted cloud logging
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "texunslotht",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = training_args,
)

# Start training
trainer_stats = trainer.train()

## 6: Expert Inference Test

Test the model inside the notebook to see if it adopts the expert persona.

In [ ]:
FastLanguageModel.for_inference(model) # 2x faster inference

prompt = "Explain the security implications of finding an exposed .env file on a public web server."

inputs = tokenizer(
[
    f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

## 7: Export to GGUF (For LM Studio/Ollama)

This merges the LoRA adapter into the base model and quantizes it into a single file.

In [ ]:
# Save the final GGUF model directly to your Google Drive
model.save_pretrained_gguf(
    "models/final_gguf/cyber_expert_gemma",
    tokenizer,
    quantization_method = "q4_k_m"
)